In [2]:
!pip install pandas

   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ----------- ---------------------------- 2.9/9.9 MB 15.2 MB/s eta 0:00:01
   ------------------------- -------------- 6.3/9.9 MB 15.4 MB/s eta 0:00:01
   --------------------------------- ------ 8.4/9.9 MB 15.3 MB/s eta 0:00:01
   ---------------------------------------- 9.9/9.9 MB 12.9 MB/s  0:00:00

   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas

In [4]:
!pip install tensorflow

   ---------------------------------------- 0.0/350.8 MB ? eta -:--:--
   ---------------------------------------- 1.8/350.8 MB 9.2 MB/s eta 0:00:38
   ---------------------------------------- 4.2/350.8 MB 10.1 MB/s eta 0:00:35
    --------------------------------------- 6.8/350.8 MB 10.8 MB/s eta 0:00:32
   - -------------------------------------- 10.0/350.8 MB 11.7 MB/s eta 0:00:30
   - -------------------------------------- 12.1/350.8 MB 11.3 MB/s eta 0:00:31
   - -------------------------------------- 15.2/350.8 MB 12.0 MB/s eta 0:00:29
   -- ------------------------------------- 18.4/350.8 MB 12.3 MB/s eta 0:00:27
   -- ------------------------------------- 21.0/350.8 MB 12.3 MB/s eta 0:00:27
   -- ------------------------------------- 24.1/350.8 MB 12.5 MB/s eta 0:00:27
   --- ------------------------------------ 27.0/350.8 MB 12.6 MB/s eta 0:00:26
   --- ------------------------------------ 30.9/350.8 MB 13.1 MB/s eta 0:00:25
   --- ------------------------------------ 33.0/350.

In [1]:
# Importación de bibliotecas

from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
import sys

from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing import image
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder

In [22]:
!{sys.executable} -m pip install pillow

In [21]:
!{sys.executable} -m pip install scikit-learn

   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   --------- ------------------------------ 1.8/8.1 MB 11.2 MB/s eta 0:00:01
   -------------------- ------------------- 4.2/8.1 MB 10.5 MB/s eta 0:00:01
   -------------------------------- ------- 6.6/8.1 MB 10.9 MB/s eta 0:00:01
   ---------------------------------------- 8.1/8.1 MB 10.2 MB/s  0:00:00

   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   ----------------------

In [12]:
# Configuración de rutas y clases

DATASET_DIR = Path(r"C:\Users\Celia\Documents\Master\2 Trimestre\TFM\miProyecto\dataset_augmented")

CLASSES = ["multirotor", "ala_fija", "helicoptero", "no_uav"]

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

FEATURES_DIR = Path(r"C:\Users\Celia\Documents\Master\2 Trimestre\TFM\miProyecto\features_resnet50")
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

In [13]:
# Creación del extractor ResNet50

extractor = ResNet50(
    weights="imagenet",
    include_top=False,
    pooling="avg",
    input_shape=(224, 224, 3)
)

extractor.trainable = False

print("Extractor ResNet50 cargado correctamente.")

Extractor ResNet50 cargado correctamente.


In [14]:
# Función para recoger rutas y etiquetas

def get_image_paths_and_labels(split_dir, classes):
    image_paths = []
    labels = []

    valid_extensions = {".jpg", ".jpeg", ".png"}

    for class_name in classes:
        class_dir = split_dir / class_name

        if not class_dir.exists():
            raise FileNotFoundError(f"No existe la carpeta: {class_dir}")

        for img_path in class_dir.iterdir():
            if img_path.suffix.lower() in valid_extensions:
                image_paths.append(img_path)
                labels.append(class_name)

    return image_paths, labels

In [15]:
# Comprabación de cuántas imágenes hay

for split in ["train", "val", "test"]:
    paths, labels = get_image_paths_and_labels(DATASET_DIR / split, CLASSES)
    print(f"{split}: {len(paths)} imágenes")

    for class_name in CLASSES:
        print(f"  {class_name}: {labels.count(class_name)}")

train: 2100 imágenes
  multirotor: 525
  ala_fija: 525
  helicoptero: 525
  no_uav: 525
val: 152 imágenes
  multirotor: 38
  ala_fija: 38
  helicoptero: 38
  no_uav: 38
test: 148 imágenes
  multirotor: 37
  ala_fija: 37
  helicoptero: 37
  no_uav: 37


In [16]:
# Función para cargar imágenes por lotes

def load_images_batch(batch_paths, img_size=(224, 224)):
    images = []

    for img_path in batch_paths:
        img = image.load_img(img_path, target_size=img_size)
        img_array = image.img_to_array(img)
        images.append(img_array)

    return np.array(images, dtype=np.float32)

In [17]:
# Función para extraer características

def extract_features(image_paths, labels, extractor, batch_size=32, img_size=(224, 224)):
    all_features = []
    all_labels = []
    all_paths = []

    for start in tqdm(range(0, len(image_paths), batch_size), desc="Extrayendo características"):
        end = start + batch_size

        batch_paths = image_paths[start:end]
        batch_labels = labels[start:end]

        # Cargar imágenes
        batch_images = load_images_batch(batch_paths, img_size)

        # Preprocesamiento específico de ResNet50
        batch_images = preprocess_input(batch_images)

        # Extraer características
        batch_features = extractor.predict(batch_images, verbose=0)

        all_features.append(batch_features)
        all_labels.extend(batch_labels)
        all_paths.extend([str(p) for p in batch_paths])

    X = np.vstack(all_features)
    y = np.array(all_labels)

    return X, y, all_paths

In [18]:
# Extraer características de train, val y test

# TRAIN
train_paths, train_labels = get_image_paths_and_labels(DATASET_DIR / "train", CLASSES)
X_train_features, y_train, train_paths_str = extract_features(
    train_paths,
    train_labels,
    extractor,
    batch_size=BATCH_SIZE,
    img_size=IMG_SIZE
)

print("Train:", X_train_features.shape, y_train.shape)

# VALIDACIÓN
val_paths, val_labels = get_image_paths_and_labels(DATASET_DIR / "val", CLASSES)
X_val_features, y_val, val_paths_str = extract_features(
    val_paths,
    val_labels,
    extractor,
    batch_size=BATCH_SIZE,
    img_size=IMG_SIZE
)

print("Validación:", X_val_features.shape, y_val.shape)

# TEST
test_paths, test_labels = get_image_paths_and_labels(DATASET_DIR / "test", CLASSES)
X_test_features, y_test, test_paths_str = extract_features(
    test_paths,
    test_labels,
    extractor,
    batch_size=BATCH_SIZE,
    img_size=IMG_SIZE
)

print("Test:", X_test_features.shape, y_test.shape)

Extrayendo características: 100%|██████████| 66/66 [07:32<00:00,  6.85s/it]


Train: (2100, 2048) (2100,)


Extrayendo características: 100%|██████████| 5/5 [00:27<00:00,  5.59s/it]


Validación: (152, 2048) (152,)


Extrayendo características: 100%|██████████| 5/5 [00:23<00:00,  4.64s/it]

Test: (148, 2048) (148,)


In [22]:
# Codificar etiquetas a números

label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded = label_encoder.transform(y_val)
y_test_encoded = label_encoder.transform(y_test)

print("Clases:", label_encoder.classes_)
print("Ejemplo etiquetas codificadas:", y_train_encoded[:10])

Clases: ['ala_fija' 'helicoptero' 'multirotor' 'no_uav']
Ejemplo etiquetas codificadas: [2 2 2 2 2 2 2 2 2 2]


In [23]:
# Guardar las características 

np.save(FEATURES_DIR / "X_train_features.npy", X_train_features)
np.save(FEATURES_DIR / "X_val_features.npy", X_val_features)
np.save(FEATURES_DIR / "X_test_features.npy", X_test_features)

np.save(FEATURES_DIR / "y_train.npy", y_train_encoded)
np.save(FEATURES_DIR / "y_val.npy", y_val_encoded)
np.save(FEATURES_DIR / "y_test.npy", y_test_encoded)

np.save(FEATURES_DIR / "classes.npy", label_encoder.classes_)

In [24]:
# Comprobar que se han guardado bien

print("Archivos guardados en:")
print(FEATURES_DIR)

for file in FEATURES_DIR.iterdir():
    print(file.name)

Archivos guardados en:
C:\Users\Celia\Documents\Master\2 Trimestre\TFM\miProyecto\features_resnet50
classes.npy
X_test_features.npy
X_train_features.npy
X_val_features.npy
y_test.npy
y_train.npy
y_val.npy


In [ ]:
# Cargar características de nuevo

X_train_features = np.load(FEATURES_DIR / "X_train_features.npy")
X_val_features = np.load(FEATURES_DIR / "X_val_features.npy")
X_test_features = np.load(FEATURES_DIR / "X_test_features.npy")

y_train_encoded = np.load(FEATURES_DIR / "y_train.npy")
y_val_encoded = np.load(FEATURES_DIR / "y_val.npy")
y_test_encoded = np.load(FEATURES_DIR / "y_test.npy")

classes = np.load(FEATURES_DIR / "classes.npy", allow_pickle=True)

print(X_train_features.shape)
print(classes)